# MyTravels GitOps Runbook (Argo CD)

This notebook stands up the same MyTravels stack as [lesson 3](<../3-kubernetes/runbook.ipynb>), but nothing is deployed with `kubectl apply` any more. Argo CD runs inside the cluster, watches [manifests/](manifests/) on the remote branch, and reconciles the cluster to match it. Run cells top to bottom.

## Summary

- **Step 1 — Prerequisites**: Verify the required tools are installed.
- **Step 2 — Create the Cluster**: Same k3d cluster as lesson 3 (1 control plane, 3 workers, ports 8080 and 5432).
- **Step 3 — /etc/hosts**: Add `*.mytravels.local` hostnames, now including `argocd.mytravels.local`.
- **Step 4 — Traefik Configuration**: Add the raw-TCP `postgres` entrypoint. Cluster bootstrap, not GitOps.
- **Step 5 — Install Argo CD**: Install the controller into the `argocd` namespace and switch the API server to HTTP.
- **Step 6 — Argo CD UI and Credentials**: Expose the UI through Traefik and set the login to `user123` / `password123`.
- **Step 7 — Argo CD CLI** *(optional)*: Log in with the `argocd` binary.
- **Step 8 — Push to Git**: Argo clones from the remote, so the manifests must be pushed before they can be deployed.
- **Step 9 — Namespace and Secrets**: The one part of the application stack applied by hand, and why.
- **Step 10 — Deploy**: Create the AppProject and Application, and watch the sync waves roll through.
- **Step 11 — Verification**: Confirm the stack is healthy and reachable.
- **Step 12 — Change Something**: Edit a manifest in git and watch Argo roll it out.
- **Step 13 — Drift and Self-Heal**: Break the cluster by hand, then let Argo put it back.
- **Step 14 — Diagnostics**: Where to look when a sync fails.
- **Step 15 — Teardown**: Remove the Application, Argo CD, and the cluster.

---

## What changed from lesson 3

The manifests are the same workloads, but a few things had to change to be deployable by a controller rather than by a human running `kubectl` in order.

| Lesson 3 | Here | Why |
|---|---|---|
| Numeric filename prefixes (`1-namespace.yaml`, `8-traefik-config.yaml`) | Plain names | `kubectl apply -f dir/` walks a directory alphabetically. Argo does not — it sorts by resource kind and by sync wave, so the numbers were meaningless and misleading. |
| Ordering by running cells in sequence | `argocd.argoproj.io/sync-wave` annotations | Argo applies every manifest at once unless told otherwise. Waves re-establish the order the runbook used to enforce by hand: namespace → data services → migrations → apps → ingress. |
| `db-migrations` Job applied like any other manifest | Job is a `Sync` hook with `hook-delete-policy: BeforeHookCreation` | A Job's pod template is immutable. As an ordinary resource the second sync would try to patch it and fail; as a hook it is recreated each sync. |
| `1-secret.yaml` per service, hand-written, gitignored | Secrets created from `.env` by [Step 9](#step-9), outside Argo | GitOps deploys what is in git, and plaintext secrets do not belong there. This is the honest version of the problem — see the note in Step 9 for the real fix. |
| `8-traefik-config.yaml` applied with the app | [cluster/traefik-config.yaml](cluster/traefik-config.yaml), applied at bootstrap | It patches a k3s-owned Helm release in `kube-system`, outside the AppProject's blast radius, and restarts Traefik. |
| `kubectl apply -f manifests/postgres` | `git push` | The cluster follows the remote branch. Local edits change nothing until they are pushed. |

**Sync waves used here**

| Wave | Resources |
|---|---|
| `-1` | `mytravels-default` namespace |
| `0` | postgres, rabbitmq, minio (PVs, PVCs, ConfigMap, Deployments, Services) |
| `1` | `db-migrations` Job (sync hook); observability stack (otel-collector, Prometheus, Tempo, postgres-exporter, cadvisor, Grafana) — no hard dependency on migrations, but both need wave 0's Services to exist |
| `2` | api, messaging, web |
| `3` | ingress |

Argo waits for every resource in a wave to report **Healthy** before starting the next one, which is what makes wave 1 safe to run against a database that only just came up.

---

## Step 1 — Prerequisites

Rancher Desktop, k3d, kubectl, JupyterLab, and Freelens/OpenLens install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

Two additions for this lesson:

- **git** with push access to this repository — Argo CD clones from the remote, so `git push` is now part of deploying.
- **argocd CLI** *(optional)* — everything below works with `kubectl` alone, but the CLI gives much better sync output. [Install instructions](https://argo-cd.readthedocs.io/en/stable/cli_installation/).

Open Rancher Desktop and make sure the container engine is running before continuing.

In [ ]:
%%bash
echo "=== Docker ==="
docker --version
echo "=== k3d ==="
k3d --version
echo "=== kubectl ==="
kubectl version --client 2>/dev/null || kubectl version --client --short
echo "=== git ==="
git --version
echo "=== argocd CLI (optional) ==="
argocd version --client 2>/dev/null || echo "not installed — optional, kubectl is enough"

---

## Step 2 — Create the Cluster

![cluster](images/k8s%20components.drawio.png)

Identical to lesson 3: 1 control plane, 3 workers, Traefik bundled by k3s.

> **The cluster name matters.** The MinIO deployment and its hostPath PVs pin themselves to `k3d-mytravels-agent-2` via `nodeSelector`, so the cluster must be named `mytravels` or MinIO will stay `Pending` forever.

> **This is the same name and the same host ports (8080, 5432) as lesson 3.** If that cluster still exists, delete it first — the next cell checks and tells you.

> **Ensure Rancher Desktop is running before this cell.** k3d creates the cluster's nodes as Docker containers, so it needs a live Docker daemon.

In [ ]:
%%bash
k3d cluster list
echo ""
if k3d cluster list -o json | grep -q '"name":"mytravels"'; then
  echo "A cluster named 'mytravels' already exists."
  echo "Either reuse it (skip the create cell) or delete it first:"
  echo "    k3d cluster delete mytravels"
else
  echo "No 'mytravels' cluster — safe to create."
fi

In [ ]:
%%bash
k3d cluster create mytravels \
  -p "8080:80@loadbalancer" \
  -p "5432:5432@loadbalancer" \
  --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
  --servers 1 \
  --agents 3

In [ ]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Traefik ==="
kubectl get pods -n kube-system -l app.kubernetes.io/name=traefik

---

## Step 3 — /etc/hosts

Same host-based routing as lesson 3, plus `argocd.mytravels.local` for the Argo CD UI.

```
127.0.0.1  rabbitmq.mytravels.local
127.0.0.1  minio.mytravels.local
127.0.0.1  api.mytravels.local
127.0.0.1  messaging.mytravels.local
127.0.0.1  web.mytravels.local
127.0.0.1  argocd.mytravels.local
```

Run **one** of the next two cells depending on your OS. If you already did lesson 3, only the `web` and `argocd` entries will be added — the others are detected and skipped.

### macOS/Linux

In [ ]:
import subprocess
import getpass

password = getpass.getpass("sudo password: ")

hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "web.mytravels.local",
    "argocd.mytravels.local",
]

with open("/etc/hosts", "r") as f:
    current = f.read()

for host in hosts:
    if host in current:
        print(f"Already present: {host}")
    else:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} — {result.stderr.strip()}")

### Windows

In [ ]:
import ctypes

hosts_path = r"C:\Windows\System32\drivers\etc\hosts"
hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "web.mytravels.local",
    "argocd.mytravels.local",
]

def is_admin():
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

if not is_admin():
    print("Not running as Administrator — the hosts file is not writable.")
    print("Close Jupyter/VS Code and relaunch it via 'Run as Administrator', then re-run this cell.")
else:
    with open(hosts_path, "r") as f:
        current = f.read()

    with open(hosts_path, "a") as f:
        for host in hosts:
            if host in current:
                print(f"Already present: {host}")
            else:
                f.write(f"127.0.0.1  {host}\n")
                print(f"Added: {host}")

---

## Step 4 — Traefik Configuration

Adds the raw-TCP `postgres` entrypoint so the `IngressRouteTCP` in [manifests/ingress.yaml](manifests/ingress.yaml) can route port 5432.

This runs **before** Argo CD and is applied with `kubectl`, not synced. Two reasons, both worth internalising:

1. It patches a Helm release owned by k3s in `kube-system` — outside the `mytravels` AppProject's destination namespace, so a sync would be rejected by design.
2. Applying it restarts Traefik, which briefly drops every ingress in the cluster including Argo CD's own UI. That is a cluster maintenance operation, not an application deploy.

Deciding what the GitOps controller owns and what the cluster bootstrap owns is a real design decision, not an implementation detail.

In [ ]:
%%bash
kubectl apply -f cluster/traefik-config.yaml
echo ""
echo "Waiting for Traefik to restart..."
sleep 20
kubectl rollout status deploy/traefik -n kube-system --timeout=60s
echo ""
for i in $(seq 1 12); do
  ARGS=$(kubectl get deploy traefik -n kube-system -o jsonpath='{.spec.template.spec.containers[0].args}' | tr ',' '\n')
  if echo "$ARGS" | grep -q postgres; then
    echo "postgres entrypoint confirmed:"
    echo "$ARGS" | grep postgres
    break
  fi
  echo "Waiting for postgres entrypoint... ($i/12)"
  sleep 5
done

---

## Step 5 — Install Argo CD

Installs the upstream manifest into the `argocd` namespace. This is the one component that cannot be managed by GitOps at install time — something has to create the controller before the controller can manage anything. (Teams often make Argo CD manage its own manifests immediately afterwards; that is the "app of apps" pattern, out of scope here.)

| Component | Role |
|---|---|
| `argocd-application-controller` | The reconciliation loop — compares git to the cluster and applies the difference |
| `argocd-repo-server` | Clones repos and renders manifests |
| `argocd-server` | API and web UI |
| `argocd-redis` | Cache for rendered manifests and the resource tree |
| `argocd-dex-server` | SSO connector (unused here) |
| `argocd-applicationset-controller` | Generates Applications from templates (unused here) |
| `argocd-notifications-controller` | Sends sync events to Slack/webhooks (unused here) |

That is roughly 1–1.5 GB of memory on top of the MyTravels stack. Fine on a normal dev machine.

Two details in the cell below:

- **`--server-side`** — the install manifest's CRDs are larger than the 256 KB limit on the `last-applied-configuration` annotation that client-side apply writes. Server-side apply has no such limit.
- **Version** — the cell resolves the latest release from the GitHub API. For a reproducible run, pin it instead by setting `ARGOCD_VERSION` to a tag from the [releases page](https://github.com/argoproj/argo-cd/releases) before running, e.g. `%env ARGOCD_VERSION=v3.0.0`.

In [ ]:
%%bash
ARGOCD_VERSION="${ARGOCD_VERSION:-$(curl -fsSL https://api.github.com/repos/argoproj/argo-cd/releases/latest \
  | grep -m1 '"tag_name"' | cut -d '"' -f 4)}"

if [ -z "$ARGOCD_VERSION" ]; then
  echo "Could not reach the GitHub API — falling back to the 'stable' branch"
  ARGOCD_VERSION=stable
fi

echo "Installing Argo CD: $ARGOCD_VERSION"
echo ""

kubectl apply -f argocd/namespace.yaml
kubectl apply --server-side -n argocd \
  -f "https://raw.githubusercontent.com/argoproj/argo-cd/${ARGOCD_VERSION}/manifests/install.yaml"

In [ ]:
%%bash
echo "Waiting for Argo CD to come up (first run pulls ~5 images)..."
kubectl rollout status statefulset/argocd-application-controller -n argocd --timeout=300s
kubectl wait --for=condition=Available deployment --all -n argocd --timeout=300s
echo ""
kubectl get pods -n argocd

### Switch the API server to HTTP

`argocd-server` serves HTTPS with a self-signed certificate and redirects plain HTTP to it. Behind Traefik's `web` (HTTP) entrypoint that produces a redirect loop in the browser, so TLS is turned off at the server and the ingress owns the edge. In a real cluster you would terminate a real certificate at the ingress instead — see [3-install certificates.md](<../3-install certificates.md>).

In [ ]:
%%bash
kubectl apply -f argocd/server-params.yaml
kubectl rollout restart deployment/argocd-server -n argocd
kubectl rollout status deployment/argocd-server -n argocd --timeout=180s
echo ""
echo "server.insecure ="
kubectl get configmap argocd-cmd-params-cm -n argocd -o jsonpath='{.data.server\.insecure}'
echo ""

---

## Step 6 — Argo CD UI and Credentials

Exposes `argocd-server` through Traefik at [http://argocd.mytravels.local:8080](http://argocd.mytravels.local:8080) and sets the login to `user123` / `password123`, matching every other service in this lesson.

**The built-in `admin` account cannot be renamed** — the name is hardcoded in Argo CD. A username of `user123` therefore means creating an *additional* local account and granting it the admin role, which is what [argocd/accounts.yaml](argocd/accounts.yaml) does:

| File | Sets |
|---|---|
| `argocd-cm` | `accounts.user123: apiKey, login` — creates the account and what it may do |
| `argocd-rbac-cm` | `g, user123, role:admin` — grants it full rights, with `policy.default: ''` denying everyone else |

The password is **not** in either ConfigMap. Argo CD stores passwords as bcrypt hashes in the `argocd-secret` Secret, and the runbook writes them there from `.env` — the same reasoning as the application secrets in Step 9. Both `user123` and `admin` get the same password so a botched change cannot lock you out of the lab; in production you would instead set `admin.enabled: "false"` once a real account works.

This also makes `argocd-initial-admin-secret` irrelevant. Argo CD expects you to delete it after setting a real password.

In [ ]:
%%bash
if [ -f .env ]; then
  echo ".env already exists — skipping"
else
  cp .env.example .env
  echo "Copied .env.example to .env — edit it with real values before continuing"
fi
echo ""
grep -E '^ARGOCD_' .env

In [ ]:
%%bash
kubectl apply -f argocd/accounts.yaml
echo ""
echo "=== accounts ==="
kubectl get configmap argocd-cm -n argocd -o jsonpath='{.data}' ; echo ""
echo "=== rbac ==="
kubectl get configmap argocd-rbac-cm -n argocd -o jsonpath='{.data.policy\.csv}' ; echo ""

In [ ]:
%%bash
set -euo pipefail

# grep + cut rather than `source .env` — see the note in Step 9 on why sourcing
# this file is unsafe.
ARGOCD_USER=$(grep -E '^ARGOCD_USER=' .env | cut -d= -f2-)
ARGOCD_PASSWORD=$(grep -E '^ARGOCD_PASSWORD=' .env | cut -d= -f2-)

if [ -z "$ARGOCD_USER" ] || [ -z "$ARGOCD_PASSWORD" ]; then
  echo "ARGOCD_USER / ARGOCD_PASSWORD missing from .env"
  exit 1
fi

# Argo CD stores passwords as bcrypt hashes. Two wrinkles:
#   - htpasswd emits the $2y$ prefix; Go's bcrypt only accepts $2a$ / $2b$, hence
#     the rewrite. ('$' is not an anchor mid-pattern, so sed matches it literally.)
#   - no local htpasswd is needed — Docker is already a prerequisite of this lesson.
if command -v htpasswd >/dev/null 2>&1; then
  HASH=$(htpasswd -nbBC 10 "" "$ARGOCD_PASSWORD" | tr -d ':\n' | sed 's/$2y/$2a/')
else
  echo "htpasswd not installed locally — hashing via the httpd image"
  HASH=$(docker run --rm httpd:2-alpine \
    htpasswd -nbBC 10 "" "$ARGOCD_PASSWORD" | tr -d ':\n' | sed 's/$2y/$2a/')
fi

# passwordMtime is not decoration: Argo CD invalidates every session token issued
# before it, so changing a password logs that account out everywhere.
NOW=$(date -u +%Y-%m-%dT%H:%M:%SZ)

kubectl -n argocd patch secret argocd-secret --type merge -p "{
  \"stringData\": {
    \"accounts.${ARGOCD_USER}.password\": \"${HASH}\",
    \"accounts.${ARGOCD_USER}.passwordMtime\": \"${NOW}\",
    \"admin.password\": \"${HASH}\",
    \"admin.passwordMtime\": \"${NOW}\"
  }
}"

echo ""
echo "Password set for '$ARGOCD_USER' and 'admin'."
echo "argocd-server picks this up within a few seconds — no restart required."

In [ ]:
%%bash
kubectl apply -f argocd/ingress.yaml
echo ""
kubectl get ingress -n argocd

In [ ]:
%%bash
ARGOCD_USER=$(grep -E '^ARGOCD_USER=' .env | cut -d= -f2-)
ARGOCD_PASSWORD=$(grep -E '^ARGOCD_PASSWORD=' .env | cut -d= -f2-)

echo "=== URL ==="
echo "http://argocd.mytravels.local:8080"
echo ""
echo "=== Credentials ==="
echo "username: $ARGOCD_USER"
echo "password: $ARGOCD_PASSWORD"
echo "(the built-in 'admin' account takes the same password)"
echo ""
echo "=== Login check ==="
# Hits the same session endpoint the UI uses, so a 200 here means the browser
# login will work — a stronger check than 'the page loads'.
for i in $(seq 1 12); do
  CODE=$(curl -s -o /dev/null -w '%{http_code}' \
    -X POST http://argocd.mytravels.local:8080/api/v1/session \
    -H 'Content-Type: application/json' \
    -d "{\"username\":\"${ARGOCD_USER}\",\"password\":\"${ARGOCD_PASSWORD}\"}")
  if [ "$CODE" = "200" ]; then
    echo "HTTP 200 — '$ARGOCD_USER' can log in"
    break
  fi
  echo "attempt $i: HTTP $CODE — waiting for argocd-server to reload the account..."
  sleep 5
done

In [ ]:
%%bash
# No longer used now that a real password is set. Argo CD recreates it only on a
# fresh install, so this is safe and is what the docs recommend.
kubectl -n argocd delete secret argocd-initial-admin-secret --ignore-not-found

---

## Step 7 — Argo CD CLI *(optional)*

Skip this if you did not install the `argocd` binary — every step below has a `kubectl` equivalent. The CLI is worth it for `argocd app diff` and `argocd app sync --dry-run`, which have no clean kubectl equivalent.

In [ ]:
%%bash
if ! command -v argocd >/dev/null 2>&1; then
  echo "argocd CLI not installed — skipping (optional)"
  exit 0
fi

ARGOCD_USER=$(grep -E '^ARGOCD_USER=' .env | cut -d= -f2-)
ARGOCD_PASSWORD=$(grep -E '^ARGOCD_PASSWORD=' .env | cut -d= -f2-)

argocd login argocd.mytravels.local:8080 \
  --username "$ARGOCD_USER" \
  --password "$ARGOCD_PASSWORD" \
  --plaintext \
  --grpc-web
echo ""
argocd account get-user-info
echo ""
argocd account list

---

## Step 8 — Push to Git

**This is the step that has no equivalent in lesson 3, and the one people forget.**

Argo CD's repo-server clones the repository from inside the cluster. It has no access to your working tree. Editing a file in [manifests/](manifests/) changes nothing until the change is committed *and pushed* to the branch named in [argocd/application.yaml](argocd/application.yaml) (`targetRevision: main`).

Argo polls the remote roughly every three minutes. Production setups replace polling with a webhook from GitHub.

> **Private repository?** The cell below only checks that the manifests are visible on the remote. If the repo is private, Argo also needs credentials — create a repository secret in the `argocd` namespace ([docs](https://argo-cd.readthedocs.io/en/stable/operator-manual/declarative-setup/#repositories)).

> **Working entirely offline?** Serve the repo from the host with `git daemon --export-all --base-path=<parent-dir>` and point `repoURL` at `git://host.k3d.internal:9418/intro-to-k8s`. k3d injects `host.k3d.internal` into every node's DNS.

In [ ]:
%%bash
BRANCH=$(git -C .. rev-parse --abbrev-ref HEAD)
TARGET=$(grep -m1 'targetRevision:' argocd/application.yaml | awk '{print $2}')
REPO=$(grep -m1 'repoURL:' argocd/application.yaml | awk '{print $2}')

echo "Application repoURL:        $REPO"
echo "Application targetRevision: $TARGET"
echo "Local branch:               $BRANCH"
echo ""

if [ "$BRANCH" != "$TARGET" ]; then
  echo "WARNING: you are on '$BRANCH' but the Application tracks '$TARGET'."
  echo "         Either switch branches or edit targetRevision in argocd/application.yaml."
  echo ""
fi

git -C .. fetch origin --quiet
LOCAL=$(git -C .. rev-parse HEAD)
REMOTE=$(git -C .. rev-parse "origin/$TARGET" 2>/dev/null)

echo "Local HEAD:        $LOCAL"
echo "origin/$TARGET:    ${REMOTE:-<branch not on remote>}"
echo ""

COUNT=$(git -C .. ls-tree -r --name-only "origin/$TARGET" -- 4-argocd/manifests 2>/dev/null | wc -l | tr -d ' ')
echo "Manifests visible to Argo on origin/$TARGET: $COUNT"

if [ "$COUNT" = "0" ]; then
  echo ""
  echo "Argo CD will find nothing to deploy. Commit and push 4-argocd/ first:"
  echo "    git add 4-argocd && git commit -m 'feat: argocd lesson' && git push origin $TARGET"
elif [ "$LOCAL" != "$REMOTE" ]; then
  echo ""
  echo "Local HEAD differs from the remote — Argo deploys the REMOTE version."
  echo "Uncommitted or unpushed manifest edits will not be applied."
  git -C .. status --short -- 4-argocd
fi

---

## Step 9 — Namespace and Secrets

Two things are applied by hand before Argo takes over.

**The namespace** — Argo would create it (it is in `manifests/namespace.yaml` at wave `-1`), but the secrets below need somewhere to live first. Applying it here is harmless: when Argo syncs it finds the namespace already matching git and reports it `Synced`. Adopting existing resources is normal Argo behaviour, not a workaround.

**The secrets** — this is the interesting one. `.gitignore` in this repo ends with `*secret.yaml`, which is correct and which breaks naive GitOps: a controller that deploys what is in git cannot deploy what is deliberately not in git. Three ways out, in increasing order of correctness:

| Approach | Trade-off |
|---|---|
| Apply secrets out-of-band (what we do here) | Honest and simple, but the cluster is no longer fully reproducible from git — one manual step stands between a clean cluster and a running stack. |
| [Sealed Secrets](https://github.com/bitnami-labs/sealed-secrets) or [SOPS](https://github.com/getsops/sops) | The *encrypted* file is committed, so git really does describe the whole system. One more controller (or a decryption key) to manage. This is the answer for most teams. |
| [External Secrets Operator](https://external-secrets.io/) pulling from Vault / AWS Secrets Manager / Azure Key Vault | Nothing sensitive touches git at all. Requires a secrets backend to already exist. |

The cell below reads the `.env` created back in Step 6 and creates seven Secrets whose keys exactly match what the deployments reference by `secretKeyRef`.

| Secret | Keys |
|---|---|
| `postgres-secret` | `POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB` |
| `rabbitmq-secret` | `RABBITMQ_DEFAULT_USER`, `RABBITMQ_DEFAULT_PASS` |
| `minio-secret` | `MINIO_ROOT_USER`, `MINIO_ROOT_PASSWORD` |
| `migrations-secret` | `ConnectionStrings__CoreDbContext` |
| `api-secret` | `ConnectionStrings__CoreDbContext`, `RabbitMQ__Uri`, `MinIO__AccessKey`, `MinIO__SecretKey` |
| `messaging-secret` | the four above, plus `ContentSafetyEndpoint`, `ContentSafetyKey` |
| `grafana-secret` | `GF_SECURITY_ADMIN_USER`, `GF_SECURITY_ADMIN_PASSWORD` — read by `manifests/observability/grafana-deployment.yaml` |

In [ ]:
%%bash
kubectl apply -f manifests/namespace.yaml
kubectl get namespace mytravels-default

In [ ]:
# Reads .env and creates the Secrets the manifests reference.
#
# Parsed in Python rather than `source .env` on purpose: CORE_DB_CONTEXT contains
# semicolons ("Host=postgres;Port=5432;..."), and bash would treat each one as a
# command separator — silently truncating the value to "Host=postgres" instead of
# failing. The connection string would then be wrong in a way that only shows up
# later as a migration error.
import pathlib
import subprocess

NAMESPACE = "mytravels-default"

env = {}
for line in pathlib.Path(".env").read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    env[key.strip()] = value.strip()


def need(*keys):
    missing = [k for k in keys if not env.get(k)]
    if missing:
        raise SystemExit(f"Missing from .env: {', '.join(missing)}")
    return {k: env[k] for k in keys}


shared = {
    "ConnectionStrings__CoreDbContext": env.get("CORE_DB_CONTEXT", ""),
    "RabbitMQ__Uri": env.get("RABBIT_MQ_URI", ""),
    "MinIO__AccessKey": env.get("MINIO_ROOT_USER", ""),
    "MinIO__SecretKey": env.get("MINIO_ROOT_PASSWORD", ""),
}

need("POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_DB",
     "RABBITMQ_DEFAULT_USER", "RABBITMQ_DEFAULT_PASS",
     "MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD",
     "CORE_DB_CONTEXT", "RABBIT_MQ_URI",
     "CONTENT_SAFETY_ENDPOINT", "CONTENT_SAFETY_KEY",
     "GRAFANA_ADMIN_USER", "GRAFANA_ADMIN_PASSWORD")

secrets = {
    "postgres-secret": need("POSTGRES_USER", "POSTGRES_PASSWORD", "POSTGRES_DB"),
    "rabbitmq-secret": need("RABBITMQ_DEFAULT_USER", "RABBITMQ_DEFAULT_PASS"),
    "minio-secret": need("MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD"),
    "migrations-secret": {"ConnectionStrings__CoreDbContext": env["CORE_DB_CONTEXT"]},
    "api-secret": dict(shared),
    "messaging-secret": {
        **shared,
        "ContentSafetyEndpoint": env["CONTENT_SAFETY_ENDPOINT"],
        "ContentSafetyKey": env["CONTENT_SAFETY_KEY"],
    },
    "grafana-secret": {
        "GF_SECURITY_ADMIN_USER": env["GRAFANA_ADMIN_USER"],
        "GF_SECURITY_ADMIN_PASSWORD": env["GRAFANA_ADMIN_PASSWORD"],
    },
}

placeholders = sorted({
    k for data in secrets.values() for k, v in data.items() if v.startswith("<YOUR_")
})
if placeholders:
    print(f"NOTE: still using .env.example placeholders for: {', '.join(placeholders)}")
    print("      Fine for a local run, but those services will not reach the real API.\n")

for name, data in secrets.items():
    args = ["kubectl", "create", "secret", "generic", name, "-n", NAMESPACE]
    for key, value in data.items():
        args += ["--from-literal", f"{key}={value}"]
    args += ["--dry-run=client", "-o", "yaml"]

    rendered = subprocess.run(args, capture_output=True, text=True, check=True).stdout
    subprocess.run(["kubectl", "apply", "-f", "-"], input=rendered, text=True, check=True)

In [ ]:
%%bash
# Prints key names only — never the values.
for s in postgres-secret rabbitmq-secret minio-secret migrations-secret api-secret messaging-secret grafana-secret; do
  echo -n "$s: "
  kubectl get secret "$s" -n mytravels-default \
    -o go-template='{{range $k, $v := .data}}{{$k}} {{end}}' 2>/dev/null \
    || echo -n "MISSING"
  echo ""
done

---

## Step 10 — Deploy

Everything from lesson 3's steps 6 through 14 — postgres, migrations, rabbitmq, minio, api, messaging, web, ingress — now happens because of these two objects.

| File | Creates |
|---|---|
| [argocd/appproject.yaml](argocd/appproject.yaml) | `AppProject/mytravels` — which repo, which namespace, which cluster-scoped kinds are permitted |
| [argocd/application.yaml](argocd/application.yaml) | `Application/mytravels` — points at `4-argocd/manifests` on the tracked branch |

The AppProject must exist first: an Application referencing a missing project is rejected.

Watch the second cell for the waves rolling through. `Progressing` on wave 0 means postgres/rabbitmq/minio pods are still starting; the `db-migrations` hook does not run until they report Healthy, which is exactly the `depends_on` behaviour lesson 3 had to hand-roll with a `pg_isready` loop.

In [ ]:
%%bash
kubectl apply -f argocd/appproject.yaml
kubectl apply -f argocd/application.yaml
echo ""
kubectl get appproject,application -n argocd

In [ ]:
%%bash
# Poll until the Application reports Synced/Healthy (first run pulls all images).
for i in $(seq 1 90); do
  SYNC=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.sync.status}' 2>/dev/null)
  HEALTH=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.health.status}' 2>/dev/null)
  PHASE=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.operationState.phase}' 2>/dev/null)
  echo "$(date +%T)  sync=${SYNC:-?}  health=${HEALTH:-?}  operation=${PHASE:-none}"
  if [ "$SYNC" = "Synced" ] && [ "$HEALTH" = "Healthy" ]; then
    echo ""
    echo "Application is Synced and Healthy."
    break
  fi
  sleep 10
done

In [ ]:
%%bash
echo "=== Resources managed by the Application ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='{range .status.resources[*]}{.kind}{"|"}{.name}{"|"}{.status}{"|"}{.health.status}{"\n"}{end}' \
  | column -t -s '|'
echo ""
echo "=== Migration hook ==="
kubectl get job db-migrations -n mytravels-default 2>/dev/null || echo "hook Job already cleaned up"

In [ ]:
%%bash
# The BeforeHookCreation delete policy leaves the last run's pod in place, so its
# logs survive until the next sync.
echo "--CLEANUP MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c cleanup-migrations 2>/dev/null || echo "(pod gone)"
echo ""
echo "--MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c migrate-core-db 2>/dev/null || echo "(pod gone)"

---

## Step 11 — Verification

Same checks as lesson 3 — the workloads are identical, only the delivery mechanism changed.

| Host | Routes to | Port | Protocol |
|---|---|---|---|
| [http://argocd.mytravels.local:8080](http://argocd.mytravels.local:8080) | `argocd-server` | 80 | HTTP |
| [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | `rabbitmq-management` | 15672 | HTTP |
| [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | `minio-console` | 9090 | HTTP |
| [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) | `api` | 5101 | HTTP |
| [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | `messaging` | 5102 | HTTP |
| [http://web.mytravels.local:8080](http://web.mytravels.local:8080) | `web` | 80 | HTTP |
| [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) | `grafana` | 3000 | HTTP |
| [http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080) | `prometheus` | 9090 | HTTP |
| `127.0.0.1:5432` (TCP) | `postgres` | 5432 | TCP |

> **The web UI loads but its API calls fail?** `VITE_API_BASE_URL` is inlined into the JavaScript bundle at image build time, so it cannot be changed by a Deployment env var or by anything in git. The published `v1.0.4` image points at `http://localhost:5101`, which only resolves while `kubectl port-forward svc/api 5101:5101 -n mytravels-default` is running. The GitOps-clean fix is to rebuild the image with `VITE_API_BASE_URL=http://api.mytravels.local:8080`, push the new tag, and commit the tag bump in [manifests/web/deployment.yaml](manifests/web/deployment.yaml) — a config change that ships as a git commit, like everything else here. The same applies to `VITE_OTEL_EXPORTER_OTLP_TRACES_ENDPOINT` (browser RUM traces).

The cell after next hits the API to generate a trace, then polls Tempo (via the Grafana datasource proxy) until it shows up — proof that metrics *and* traces both made it through otel-collector end to end.

In [ ]:
%%bash
echo "=== Pods ==="
kubectl get pods -n mytravels-default
echo ""
echo "=== PVCs ==="
kubectl get pvc -n mytravels-default
echo ""
echo "=== Services ==="
kubectl get svc -n mytravels-default
echo ""
echo "=== Ingress ==="
kubectl get ingress -n mytravels-default

In [ ]:
%%bash
echo "=== Argo CD ==="
curl -s -o /dev/null -w "%{http_code}" http://argocd.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== RabbitMQ Management ==="
curl -s -o /dev/null -w "%{http_code}" http://rabbitmq.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== MinIO Console ==="
curl -s -o /dev/null -w "%{http_code}" http://minio.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== API ==="
curl -s -o /dev/null -w "%{http_code}" http://api.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Messaging ==="
curl -s -o /dev/null -w "%{http_code}" http://messaging.mytravels.local:8080/health && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Web ==="
curl -s -o /dev/null -w "%{http_code}" http://web.mytravels.local:8080 && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Grafana ==="
curl -s -o /dev/null -w "%{http_code}" http://grafana.mytravels.local:8080/api/health && echo " OK" || echo " UNREACHABLE"
echo ""
echo "=== Prometheus ==="
curl -s -o /dev/null -w "%{http_code}" http://prometheus.mytravels.local:8080/-/healthy && echo " OK" || echo " UNREACHABLE"
echo ""

In [ ]:
%%bash
echo "=== Prometheus scrape targets ==="
curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
for t in data['data']['activeTargets']:
    print(t['labels'].get('job'), t['health'], t.get('lastError', ''))
"
echo ""
echo "=== Generating a trace (hit the API) ==="
curl -s -o /dev/null -w "API HTTP %{http_code}\n" http://api.mytravels.local:8080/api/pointofinterest
echo ""
echo "=== Waiting for the trace to reach Tempo (via Grafana datasource proxy) ==="
for i in $(seq 1 12); do
  RESULT=$(curl -s -u "$(grep -E '^GRAFANA_ADMIN_USER=' .env | cut -d= -f2-):$(grep -E '^GRAFANA_ADMIN_PASSWORD=' .env | cut -d= -f2-)" \
    "http://grafana.mytravels.local:8080/api/datasources/proxy/uid/tempo/api/search?tags=service.name%3Dmytravels-api&limit=1")
  COUNT=$(echo "$RESULT" | python3 -c "import json,sys; print(len(json.load(sys.stdin).get('traces', [])))" 2>/dev/null || echo 0)
  if [ "$COUNT" -gt 0 ]; then
    echo "Trace found:"
    echo "$RESULT" | python3 -m json.tool
    break
  fi
  echo "attempt $i: no trace yet, waiting..."
  sleep 5
done

---

## Step 12 — Change Something

The point of the whole exercise. Change a manifest in git and let the controller do the deploy.

1. Edit [manifests/api/deployment.yaml](manifests/api/deployment.yaml) and change `replicas: 1` to `replicas: 2`.
2. Commit and push (the next cell does both).
3. Trigger a refresh instead of waiting up to three minutes for the poll.
4. Watch the rollout.

Note what you *don't* do: no `kubectl apply`, no `kubectl set image`, no ssh into anything. The cluster state is a function of the branch. This is also why rollback is `git revert` — there is no separate deploy history to reason about.

In [ ]:
%%bash
# Scoped to 4-argocd/manifests so this never sweeps up unrelated staged work.
if git -C .. diff --quiet HEAD -- 4-argocd/manifests; then
  echo "No manifest changes — edit manifests/api/deployment.yaml first (replicas: 1 -> 2)"
  exit 0
fi

git -C .. diff --stat HEAD -- 4-argocd/manifests
echo ""
git -C .. commit -q -m "chore: scale api to 2 replicas" -- 4-argocd/manifests
git -C .. push -q origin HEAD
echo "Pushed $(git -C .. rev-parse --short HEAD)"

In [ ]:
%%bash
# A 'hard' refresh makes Argo re-clone the repo immediately instead of waiting for
# the ~3 minute poll. In production this is what a GitHub webhook triggers.
kubectl patch application mytravels -n argocd --type merge \
  -p '{"metadata":{"annotations":{"argocd.argoproj.io/refresh":"hard"}}}'

for i in $(seq 1 30); do
  SYNC=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.sync.status}')
  REV=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.sync.revision}' | cut -c1-7)
  WANT=$(kubectl get deploy api -n mytravels-default -o jsonpath='{.spec.replicas}')
  READY=$(kubectl get deploy api -n mytravels-default -o jsonpath='{.status.readyReplicas}')
  echo "$(date +%T)  sync=$SYNC  revision=$REV  api replicas=${READY:-0}/${WANT}"
  if [ "$SYNC" = "Synced" ] && [ "${READY:-0}" = "$WANT" ]; then
    echo ""
    echo "Rolled out at revision $REV — no kubectl apply involved."
    break
  fi
  sleep 6
done
echo ""
kubectl get pods -n mytravels-default -l app=api

---

## Step 13 — Drift and Self-Heal

The Application was created with `selfHeal: false` on purpose, so that up to now `kubectl edit` still behaved the way it did in lesson 3. Here is what changes when you turn it on.

**First, with self-heal off:** scale the deployment by hand. Argo notices — the Application goes `OutOfSync` — but does nothing about it. This is the "detect but don't act" mode, which is a reasonable production setting when humans sometimes need to intervene during an incident.

**Then, with self-heal on:** the same manual change is reverted within seconds. The cluster stops being something you can meaningfully edit by hand, which is the whole point and also the thing that surprises people at 2am.

In [ ]:
%%bash
echo "=== Scaling api to 3 replicas by hand ==="
kubectl scale deployment/api -n mytravels-default --replicas=3
sleep 5

kubectl patch application mytravels -n argocd --type merge \
  -p '{"metadata":{"annotations":{"argocd.argoproj.io/refresh":"normal"}}}' >/dev/null
sleep 10

echo ""
echo "Application status:"
kubectl get application mytravels -n argocd
echo ""
echo "Deployment (Argo sees the drift but leaves it alone):"
kubectl get deploy api -n mytravels-default

In [ ]:
%%bash
# Also change selfHeal to true in argocd/application.yaml and commit it — otherwise
# this patch is itself undocumented drift, in the config of the tool whose job is to
# eliminate drift.
kubectl patch application mytravels -n argocd --type merge \
  -p '{"spec":{"syncPolicy":{"automated":{"prune":true,"selfHeal":true}}}}'

echo ""
echo "=== Scaling to 3 by hand again ==="
kubectl scale deployment/api -n mytravels-default --replicas=3

for i in $(seq 1 20); do
  SPEC=$(kubectl get deploy api -n mytravels-default -o jsonpath='{.spec.replicas}')
  SYNC=$(kubectl get application mytravels -n argocd -o jsonpath='{.status.sync.status}')
  echo "$(date +%T)  spec.replicas=$SPEC  sync=$SYNC"
  sleep 5
done

---

## Step 14 — Diagnostics

Argo CD adds a layer, so failures now come in two flavours: **the sync failed** (git, RBAC, or invalid YAML — look at Argo) and **the sync succeeded but the workload is broken** (look at the pods, exactly as in lesson 3).

| Symptom | Look here |
|---|---|
| `ComparisonError` / `Unknown` sync status | repo-server logs — clone failure, bad path, missing credentials |
| `SyncFailed` with "not permitted in project" | the AppProject whitelist in [argocd/appproject.yaml](argocd/appproject.yaml) |
| Stuck `Progressing` | the pods themselves — an image pull or crash loop, not an Argo problem |
| Permanently `OutOfSync` on an unedited resource | a controller is writing a field that is not in git — needs `ignoreDifferences` |
| Job/hook errors on re-sync | the hook annotations on [manifests/migrations/job.yaml](manifests/migrations/job.yaml) |

In [ ]:
%%bash
echo "=== Application conditions ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='{range .status.conditions[*]}{.type}: {.message}{"\n"}{end}'
echo ""
echo "=== Last sync operation ==="
kubectl get application mytravels -n argocd -o jsonpath='{.status.operationState.phase}: {.status.operationState.message}'
echo ""
echo ""
echo "=== Out-of-sync resources ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='{range .status.resources[?(@.status!="Synced")]}{.kind}/{.name}: {.status}{"\n"}{end}' \
  || echo "(all synced)"

In [ ]:
%%bash
echo "=== application-controller (reconciliation) ==="
kubectl logs -n argocd statefulset/argocd-application-controller --tail=30
echo ""
echo "=== repo-server (git clone / manifest rendering) ==="
kubectl logs -n argocd deployment/argocd-repo-server --tail=20

In [ ]:
%%bash
if command -v argocd >/dev/null 2>&1; then
  echo "=== Live vs git ==="
  argocd app diff mytravels --grpc-web || true
  echo ""
  echo "=== Sync history ==="
  argocd app history mytravels --grpc-web
else
  echo "argocd CLI not installed — the UI's DIFF and HISTORY tabs show the same thing:"
  echo "http://argocd.mytravels.local:8080/applications/mytravels"
fi

In [ ]:
%%bash
for app in postgres rabbitmq minio api messaging web otel-collector prometheus tempo postgres-exporter cadvisor grafana; do
  echo "=== $app ==="
  kubectl logs -n mytravels-default -l app=$app --tail=15 2>/dev/null || echo "(no pods)"
  echo ""
done

In [ ]:
%%bash
echo "=== mytravels-default events ==="
kubectl get events -n mytravels-default --sort-by='.lastTimestamp' | tail -20
echo ""
echo "=== argocd events ==="
kubectl get events -n argocd --sort-by='.lastTimestamp' | tail -15

---

## Step 15 — Teardown

Run cells individually to tear down selectively, or run all to wipe everything.

Deleting the Application deletes everything it created — that is the `resources-finalizer.argocd.argoproj.io` finalizer doing its job. There is no `kubectl delete -f` in reverse order any more, because Argo knows what it owns.

The secrets from Step 9 are *not* owned by Argo, so they survive the Application delete. Deleting the namespace takes them with it.

In [ ]:
%%bash
# Cascades to every resource the Application created, in reverse wave order.
kubectl delete application mytravels -n argocd --wait=true --timeout=180s
kubectl delete appproject mytravels -n argocd --ignore-not-found
echo ""
kubectl get all -n mytravels-default

In [ ]:
%%bash
# Removes the hand-applied secrets along with anything left behind.
kubectl delete namespace mytravels-default --ignore-not-found

In [ ]:
%%bash
kubectl delete -f argocd/ingress.yaml --ignore-not-found
kubectl delete namespace argocd --ignore-not-found
kubectl delete crd applications.argoproj.io applicationsets.argoproj.io appprojects.argoproj.io --ignore-not-found

In [ ]:
%%bash
# Delete the entire cluster — removes all Docker containers and volumes
k3d cluster delete mytravels